# 12 Silver Claim Clean

## Purpose

This notebook creates the Silver Claim table from raw FHIR Claim resources.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.claim_raw`
2. Extract claim-level healthcare billing fields
3. Flatten nested FHIR claim structures
4. Clean patient and encounter IDs
5. Extract insurance and payment information
6. Save the clean table into the Silver layer

## Why We Are Doing This

Claim data is important for:
- healthcare cost analytics
- payer/provider analytics
- utilization analysis
- reimbursement analytics
- population health economics
- downstream ML feature engineering

## Expected Final Output

A clean Delta table:

`healthcare_catalog.silver.claim_clean`

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
We need Spark functions to flatten nested FHIR Claim resources.

### Expected Output
PySpark functions available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Claim Table

### What We Are Doing
We are loading the raw Claim table from the Bronze layer.

### Why We Are Doing This
Bronze stores raw nested FHIR Claim resources.

### Expected Output
A DataFrame named `claim_raw_df`.

In [0]:
claim_raw_df = spark.table(
    "healthcare_catalog.bronze.claim_raw"
)

print("Bronze claim_raw table loaded successfully.")

Bronze claim_raw table loaded successfully.


## Step 3 — Inspect Claim Schema

### What We Are Doing
We are printing the Claim schema.

### Why We Are Doing This
FHIR Claim resources are deeply nested and contain:
- billing information
- encounter linkage
- diagnosis linkage
- insurance details
- payment amounts

We need to inspect the structure before flattening.

### Expected Output
FHIR Claim schema.

In [0]:
claim_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Claim Columns

### What We Are Doing
We are extracting important claim-level fields from nested FHIR Claim resources.

### Why We Are Doing This
Analytics-ready healthcare billing tables require flattened structured columns.

### Fields We Will Extract

- claim_id
- patient_reference
- encounter_reference
- claim_status
- claim_use
- claim_type
- insurance_provider
- billing_start
- billing_end
- total_claim_amount
- payment_amount
- diagnosis_reference
- procedure_reference

### Expected Output
A clean DataFrame named:

`claim_clean_df`

In [0]:
claim_clean_df = claim_raw_df.select(

    col("resource.id").alias("claim_id"),

    col("resource.patient.reference").alias("patient_reference"),

    col("resource.item")[0]["encounter"][0]["reference"].alias("encounter_reference"),

    col("resource.status").alias("claim_status"),

    col("resource.use").alias("claim_use"),

    col("resource.type").alias("claim_type"),

    col("resource.insurer.display").alias("insurance_provider"),

    col("resource.billablePeriod.start").alias("billing_start"),

    col("resource.billablePeriod.end").alias("billing_end"),

    get_json_object(col("resource.total"), "$.value").cast("double").alias("total_claim_amount"),

    get_json_object(col("resource.total"), "$.currency").alias("claim_currency"),

    col("resource.payment.amount.value").cast("double").alias("payment_amount"),

    col("resource.diagnosis")[0]["diagnosisReference"]["reference"].alias("diagnosis_reference"),

    col("resource.procedure")[0]["procedureReference"]["reference"].alias("procedure_reference")
)

print("Claim clean DataFrame created successfully.")

Claim clean DataFrame created successfully.


## Step 5 — Convert Billing Dates

### What We Are Doing
We are converting billing period dates into Spark timestamp format.

### Why We Are Doing This
Timestamp conversion supports:
- healthcare utilization timelines
- financial reporting
- payer analytics
- temporal ML features

### Expected Output
Billing timestamps converted successfully.

In [0]:
claim_clean_df = claim_clean_df.withColumn(
    "billing_start",
    to_timestamp(col("billing_start"))
)

claim_clean_df = claim_clean_df.withColumn(
    "billing_end",
    to_timestamp(col("billing_end"))
)

print("Claim billing timestamps converted successfully.")

Claim billing timestamps converted successfully.


## Step 6 — Extract Clean IDs

### What We Are Doing
We are extracting clean IDs from FHIR references.

### Why We Are Doing This
Clean IDs are required for joining Claim data with:
- Patient
- Encounter
- Condition
- Procedure

tables.

### Expected Output
New columns:
- patient_id
- encounter_id
- diagnosis_condition_id
- procedure_id

In [0]:
claim_clean_df = claim_clean_df.withColumn(
    "patient_id",
    regexp_extract(col("patient_reference"), r"urn:uuid:(.*)", 1)
)

claim_clean_df = claim_clean_df.withColumn(
    "encounter_id",
    regexp_extract(col("encounter_reference"), r"urn:uuid:(.*)", 1)
)

claim_clean_df = claim_clean_df.withColumn(
    "diagnosis_condition_id",
    regexp_extract(col("diagnosis_reference"), r"urn:uuid:(.*)", 1)
)

claim_clean_df = claim_clean_df.withColumn(
    "procedure_id",
    regexp_extract(col("procedure_reference"), r"urn:uuid:(.*)", 1)
)

print("Claim IDs extracted successfully.")

Claim IDs extracted successfully.


## Step 7 — Calculate Claim Duration

### What We Are Doing
We are calculating claim billing duration in days.

### Why We Are Doing This
This can become an important Gold-layer utilization feature.

### Expected Output
A new column:

`claim_duration_days`

In [0]:
claim_clean_df = claim_clean_df.withColumn(
    "claim_duration_days",
    datediff(col("billing_end"), col("billing_start"))
)

print("Claim duration calculated successfully.")

Claim duration calculated successfully.


## Step 8 — Inspect Clean Claim Data

### What We Are Doing
We are displaying the clean claim table.

### Why We Are Doing This
We need to verify:
- claim amounts
- payer information
- patient linkage
- diagnosis linkage
- procedure linkage

### Expected Output
A clean claim-level billing table.

In [0]:
display(claim_clean_df)

claim_id,patient_reference,encounter_reference,claim_status,claim_use,claim_type,insurance_provider,billing_start,billing_end,total_claim_amount,claim_currency,payment_amount,diagnosis_reference,procedure_reference,patient_id,encounter_id,diagnosis_condition_id,procedure_id,claim_duration_days
82598759-afcb-b6f0-79d7-5edb8fb4b8d6,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:c9e46f05-f3d9-7ea6-8222-352c77ca599e,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",null,2005-11-20T08:59:49.000Z,2005-11-20T09:14:49.000Z,0.01,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,c9e46f05-f3d9-7ea6-8222-352c77ca599e,null,null,0
3fa4e72e-4294-b9ac-2893-5d87971bb549,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:c9e46f05-f3d9-7ea6-8222-352c77ca599e,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,2005-11-20T08:59:49.000Z,2005-11-20T09:14:49.000Z,1494.62,USD,null,urn:uuid:b0d31f30-61db-d170-977f-a92624fb9047,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,c9e46f05-f3d9-7ea6-8222-352c77ca599e,b0d31f30-61db-d170-977f-a92624fb9047,null,0
4eafbf45-fc20-f19c-784b-290948e07167,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:7ba6225d-d05e-581d-3a0b-2f284330a311,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",null,2005-12-20T08:59:49.000Z,2005-12-20T09:14:49.000Z,0.02,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,7ba6225d-d05e-581d-3a0b-2f284330a311,null,null,0
141cbcc2-3e99-f254-e963-770eb7051068,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:7ba6225d-d05e-581d-3a0b-2f284330a311,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,2005-12-20T08:59:49.000Z,2005-12-20T09:14:49.000Z,77.49,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,7ba6225d-d05e-581d-3a0b-2f284330a311,null,null,0
dd711ea4-a858-f664-1285-d1b1bc5967e2,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:bf353ac0-4206-79d5-0a50-4b3d54d981fd,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",null,2006-02-18T08:59:49.000Z,2006-02-18T09:14:49.000Z,0.02,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,bf353ac0-4206-79d5-0a50-4b3d54d981fd,null,null,0
9aa80977-690f-22c2-edc6-6e1de5ea1f57,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:bf353ac0-4206-79d5-0a50-4b3d54d981fd,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,2006-02-18T08:59:49.000Z,2006-02-18T09:14:49.000Z,77.49,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,bf353ac0-4206-79d5-0a50-4b3d54d981fd,null,null,0
fb7f94a6-0ee4-99b3-d911-851fe66efd7c,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:ffa64a3f-808f-d743-3719-ac74786d57b1,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",null,2006-11-26T08:59:49.000Z,2006-11-26T09:14:49.000Z,0.01,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,ffa64a3f-808f-d743-3719-ac74786d57b1,null,null,0
539d1af4-6b45-a975-079b-1ec929c21c9f,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:ffa64a3f-808f-d743-3719-ac74786d57b1,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",null,2006-11-26T08:59:49.000Z,2006-11-26T09:14:49.000Z,0.02,USD,null,null,null,7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,ffa64a3f-808f-d743-3719-ac74786d57b1,null,null,0
803eb430-77d0-5112-2abe-b0648fa2758d,urn:uuid:7fea6a12-aae3-a4d5-4abf-d6b622deaeb1,urn:uuid:ffa64a3f-808f-d743-3719-ac74786d57b1,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",null,2006-11-26T08:59:49.000Z,2006-11-26T09:14:49.000Z,0.02,USD,null,

## Step 9 — Check Claim Status Distribution

### What We Are Doing
We are counting claim records by status.

### Why We Are Doing This
This validates healthcare claim processing status.

### Expected Output
Claim status frequency table.

In [0]:
display(
    claim_clean_df.groupBy(
        "claim_status"
    ).count().orderBy(
        desc("count")
    )
)

claim_status,count
active,52068


## Step 10 — Check Claim Type Distribution

### What We Are Doing
We are counting healthcare claim types.

### Why We Are Doing This
This supports healthcare payer/provider analytics.

### Expected Output
Claim type frequency table.

In [0]:
display(
    claim_clean_df.groupBy(
        "claim_type"
    ).count().orderBy(
        desc("count")
    )
)

claim_type,count
"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",27812
"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""pharmacy""}]}",24256


## Step 11 — Calculate Total Healthcare Cost

### What We Are Doing
We are summing all healthcare claim amounts.

### Why We Are Doing This
This provides an early healthcare cost analytics metric.

### Expected Output
Total claim amount across all claims.

In [0]:
display(
    claim_clean_df.select(
        sum("total_claim_amount").alias("total_healthcare_cost")
    )
)

total_healthcare_cost
1.3353580941999365E8


## Step 12 — Check Null Values

### What We Are Doing
We are checking missing values in the claim table.

### Why We Are Doing This
Silver validation ensures high-quality analytics-ready billing data.

### Expected Output
A null-count summary table.

In [0]:
display(
    claim_clean_df.select(
        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)
            for column_name in claim_clean_df.columns
        ]
    )
)

claim_id,patient_reference,encounter_reference,claim_status,claim_use,claim_type,insurance_provider,billing_start,billing_end,total_claim_amount,claim_currency,payment_amount,diagnosis_reference,procedure_reference,patient_id,encounter_id,diagnosis_condition_id,procedure_id,claim_duration_days
0,0,0,0,0,0,52068,0,0,0,0,52068,39902,39361,0,0,39902,39361,0


## Step 13 — Save Silver Claim Table

### What We Are Doing
We are saving the clean claim table into the Silver layer.

### Why We Are Doing This
This creates reusable healthcare financial analytics data.

### Expected Output
A Delta table:

`healthcare_catalog.silver.claim_clean`

In [0]:
claim_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.claim_clean")

print("Silver claim_clean table saved successfully.")

Silver claim_clean table saved successfully.


## Step 14 — Verify Silver Tables

### What We Are Doing
We are listing all Silver tables.

### Why We Are Doing This
We want to confirm that `claim_clean` was saved successfully.

### Expected Output
`claim_clean` should appear in the Silver table list.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|silver  |allergy_intolerance_clean|false      |
|silver  |careplan_clean           |false      |
|silver  |claim_clean              |false      |
|silver  |condition_clean          |false      |
|silver  |encounter_clean          |false      |
|silver  |immunization_clean       |false      |
|silver  |medication_request_clean |false      |
|silver  |observation_clean        |false      |
|silver  |patient_clean            |false      |
|silver  |procedure_clean          |false      |
+--------+-------------------------+-----------+

